# C3: Multiscale NPPeco Synthesis

Compute R1/R3/R6/R12 rolling sums of NPPact and HANPPharv and export one NetCDF for each window.

- Input: `output/monthly/NPPact.nc`, `output/monthly/HANPPharv.nc`
- Outputs: `output/monthly/rolling_R1.nc`, `rolling_R3.nc`, `rolling_R6.nc`, and `rolling_R12.nc`
- Variables in each file: `NPPact`, `HANPPharv`, `NPPeco`, and `HANPP`
- Window sums: plain float64 sum of the `w` monthly values ending at each month, NaN at endpoints
- Intensity masked to NaN where the denominator ≤ 0

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import rioxarray
import matplotlib.pyplot as plt

import sys
sys.path.append(str(Path('../src')))

with open(Path('../config.yaml')) as f:
    cfg = yaml.safe_load(f)

In [ ]:
for k, v in cfg['paths'].items():
    cfg['paths'][k] = Path(v)
cfg['data_root'] = Path(cfg['data_root'])

## 1. Load monthly NPPact / HANPPharv

In [ ]:
da_NPPact    = xr.open_dataarray(cfg['paths']['output'] / 'monthly/NPPact.nc',    chunks={"time": 12}).astype(np.float32)
da_NPPpot    = xr.open_dataarray(cfg['paths']['output'] / 'monthly/NPPpot.nc',    chunks={"time": 12}).astype(np.float32)
da_HANPPharv = xr.open_dataarray(cfg['paths']['output'] / 'monthly/HANPPharv.nc', chunks={"time": 12}).astype(np.float32)
print('NPPact:   ', da_NPPact.dims,    da_NPPact.shape,    da_NPPact.dtype)
print('HANPPharv:', da_HANPPharv.dims, da_HANPPharv.shape, da_HANPPharv.dtype)

## 2. Compute multiscale variables

In [ ]:
# windows = {'R1': 1, 'R3': 3, 'R6': 6, 'R12': 12}
# out_path.parent.mkdir(parents=True, exist_ok=True)

# windows = {f'R{i}': i for i in range(1, 13)}
windows = {f'R{i}': i for i in [1, 3, 6, 12]}

for i, (tag, w) in enumerate(windows.items()):
    out_path = cfg['paths']['output'] / f'monthly/rolling_{tag}.nc'
    if out_path.exists():
        print(f'{tag} (window={w}) already exists, skipping.')
        continue
    
    # Plain float64 sums over each window (construct + sum). A running float32 rolling sum leaves about -4e-7
    # where production and harvest are both exactly zero, which would then be counted as a deficit.
    def window_sum(da):
        return da.astype(np.float64).rolling(time=w, center=False, min_periods=w).construct('win').sum('win', skipna=False)

    nppact_w    = window_sum(da_NPPact)
    hanppharv_w = window_sum(da_HANPPharv)
    npppot_w    = window_sum(da_NPPpot)
    nppeco_w    = nppact_w - hanppharv_w
    hanpp_w     = npppot_w - nppeco_w

    ds_w = xr.Dataset({
        'NPPact':    nppact_w,
        'HANPPharv': hanppharv_w,
        'NPPeco':    nppeco_w,
        'HANPP':     hanpp_w,
    })
    compress_kwds = {'dtype': 'float32', 'zlib': True, 'complevel': 5}
    ds_w.to_netcdf(out_path, encoding={
        'NPPact':    compress_kwds,
        'HANPPharv': compress_kwds,
        'NPPeco':    compress_kwds,
        'HANPP':     compress_kwds,
    })
